In [1]:
import os
import sys
from pathlib import Path

# Make the lane_queues package importable however Jupyter was launched
# (poetry, JupyterLab in Docker, from the repo root, ...).
_cwd = Path(os.getcwd()).resolve()
_ROOT = next(
    (p for p in [_cwd, _cwd.parent, _cwd / 'LaneQueueSimulations']
     if (p / 'lane_queues').is_dir()),
    None,
)
if _ROOT is None:
    raise RuntimeError(f"Cannot locate lane_queues from {_cwd}.")
sys.path.insert(0, str(_ROOT))

import pandas as pd
import matplotlib
%matplotlib inline
from matplotlib import pylab as plt
import multiprocessing as mp
import warnings

from lane_queues.sim_worker import *

In [ ]:
warnings.filterwarnings('ignore')
pool = mp.Pool(mp.cpu_count())

total_time = 45000
p_shopper = 0.98
p_checkout = 0.15
lines = 7

sl = simulation_lines(total_time, lines) 
spa = simulation_prob_arrive(total_time, p_shopper)
spc = simulation_prob_checkout(total_time, lines, p_checkout)
inputs2 = tuple([(i, lines, total_time, sl, spa, spc) for i in range(1, lines+1)])
#result = pool.map(lockstep_worker, inputs2)

inputs1 = tuple([(i, lines, total_time, p_shopper, p_checkout) for i in range(1, lines+1)])
result = pool.map(random_worker, inputs1)

vs = []
vw = []
ds = []
for df, d in result:
    dq = df.describe()
    vs.append(dq["avg_shoppers"]["mean"])
    vw.append(dq["avg_wait"]["mean"])
    ds.append(d)
    df.plot(x=0, y=1, title="lines to inspect:{}".format(d))
    plt.show()
    
plt.semilogy(ds, vs, "r-d")
plt.semilogy(ds, vw, "b-s")
plt.show()